# CS 5326 – PA2: Task 7 – pairwise evaluation with a Kaggle model judge

This notebook follows manual §13: it judges **saved pairwise JSONL files only** (it never regenerates model outputs),
using `kaggle_benchmarks` model-proxy access.

**Setup (manual §13.1)**
1. Create this notebook as a Kaggle **benchmark** notebook so model-proxy access is enabled
   (`MODEL_PROXY_URL` / `MODEL_PROXY_API_KEY` are set). No GPU is needed. Internet must be on so the repo can be cloned.
2. *Recommended:* **Add Input** → the `benchmark-inputs` dataset (generations + `eval/*_pairs.jsonl`). Without an input,
   the pair files committed in the GitHub repo (`outputs/eval/`) are used; they are the same files.
3. *Optional:* attach a previous version of this notebook's output to resume judging without re-spending calls.
4. **Run All**, then **Save Version** when it finishes.

**Reliability (manual §13.2):** 5 pairs per request · ≥ 5 s sleep between attempts · up to 3 attempts per batch ·
judgments rewritten after every batch · unresolved items in `*_failures.jsonl` · resume from existing judgments ·
win / loss / tie / unknown / failure summaries. The judge sees only the instruction and "Response A" / "Response B";
the A/B → model mapping is applied after the verdict returns.

| Pair file | Rubric | Comparison |
|---|---|---|
| `hh_dpo_vs_sft_pairs.jsonl` | safety | HH-RLHF DPO vs SFT on the synthetic helpful/harmless set (300) |
| `ultrafeedback_dpo_vs_sft_pairs.jsonl` | general | UltraFeedback DPO vs SFT on AlpacaEval 300 / seed 42 |
| `persona_lora_vs_base_pairs.jsonl` | persona_style **and** general | LoRA-DPO vs base SmolLM2 (style faithfulness vs usefulness/safety) |
| `sft_vs_base_alpaca_pairs.jsonl` | general | optional extra analysis (set `INCLUDE_OPTIONAL = False` to skip) |

In [ ]:
REPO_URL = "https://github.com/alihasnaink/model-alignment.git"
REPO_BRANCH = "main"
REPO_DIR = "/kaggle/working/model-alignment"

JUDGE_MODEL = "google/gemini-3.6-flash"   # manual's suggested judge; falls back to Kaggle's default judge if unavailable
PILOT_PAIRS = 3          # judge the first N pairs of each file first (in eval_pilot/) to check parsing; 0 = skip
BATCH_SIZE, SLEEP_SECONDS, MAX_RETRIES = 5, 5, 3
INCLUDE_OPTIONAL = True  # also judge SFT vs base on AlpacaEval (not required by the manual)

In [ ]:
import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
from IPython.display import FileLink, Image, Markdown, display

if not Path(REPO_DIR, "scripts").exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
repo = Path(REPO_DIR)
os.chdir(repo)
print("Repo commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
ENV = {**os.environ, "PYTHONPATH": str(repo), "PYTHONUNBUFFERED": "1"}


def run(cmd):
    # Stream a script's output into this cell (plain subprocess output would not appear in a notebook).
    # "python" -> this kernel's interpreter, so scripts see the same packages (kaggle_benchmarks) as the notebook.
    cmd = [sys.executable if c == "python" else str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    with subprocess.Popen(cmd, cwd=repo, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True)
    if proc.returncode != 0:
        raise RuntimeError(f"{cmd[1]} failed with exit code {proc.returncode}")

## 1. Check model-proxy access and pick the judge model

In [ ]:
import kaggle_benchmarks as kbench

assert kbench.kaggle.is_configured(), (
    "No model-proxy credentials (MODEL_PROXY_URL / MODEL_PROXY_API_KEY). Create this notebook as a Kaggle benchmark "
    "notebook so kaggle_benchmarks can call judge models.")
print("Models listed by the proxy:", sorted(kbench.llms) or "(none listed)")
print("Kaggle default judge:", os.environ.get("LLM_DEFAULT_JUDGE"), "| default model:", os.environ.get("LLM_DEFAULT"))

candidates = [JUDGE_MODEL, os.environ.get("LLM_DEFAULT_JUDGE"), os.environ.get("LLM_DEFAULT")]
chosen, JUDGE_TEMPERATURE = None, "0"
for name in dict.fromkeys(c for c in candidates if c):
    llm = kbench.llms.get(name) or kbench.kaggle.load_model(name)
    # Prefer temperature 0 (deterministic, reproducible judging); fall back to the provider default if rejected.
    for temperature in ("0", "none"):
        llm.support_temperature = temperature != "none"
        try:
            with kbench.chats.new(name="judge-preflight", orphan=True):
                reply = str(llm.prompt("Reply with the single word OK.", temperature=0))
            print(f"{name} (temperature {temperature}): OK -> {reply.strip()[:40]!r}")
            chosen, JUDGE_TEMPERATURE = name, temperature
            break
        except Exception as error:
            print(f"{name} (temperature {temperature}): failed ({type(error).__name__}: {str(error)[:200]})")
            time.sleep(SLEEP_SECONDS)
    if chosen:
        break
assert chosen, "No judge model answered. Check the benchmark notebook setup and the model list above."
if chosen != JUDGE_MODEL:
    print(f"NOTE: {JUDGE_MODEL} unavailable - using {chosen}. Record this judge model in the report (manual §13).")
JUDGE_MODEL = chosen
print(f"Judge: {JUDGE_MODEL} | temperature: {'provider default' if JUDGE_TEMPERATURE == 'none' else 0}")

## 2. Gather the saved pairwise files (never regenerate outputs here)

In [ ]:
OUT = Path("/kaggle/working/judge_outputs/outputs")
# Start from the repo's committed outputs (training metrics, configs, generations, pairs) for the summary tables ...
shutil.copytree(repo / "outputs", OUT, dirs_exist_ok=True)
# ... then overlay an attached benchmark-inputs dataset, which is the manual's recommended source of the pair files.
attached = sorted(Path("/kaggle/input").rglob("eval/*_pairs.jsonl")) if Path("/kaggle/input").exists() else []
if attached:
    source = attached[0].parent.parent
    for sub in ("eval", "generations"):
        if (source / sub).exists():
            shutil.copytree(source / sub, OUT / sub, dirs_exist_ok=True)
    print("Pair files from attached input:", source)
else:
    print("No input attached - using the pair files committed in the repo:", repo / "outputs" / "eval")
# Resume: reuse judgments from an attached previous run of this notebook (only files we do not already have).
for previous in (Path("/kaggle/input").rglob("eval/*_judged*.json*") if Path("/kaggle/input").exists() else []):
    dest = OUT / "eval" / previous.name
    if not dest.exists():
        shutil.copy2(previous, dest)
        print("resuming from", previous)
EVAL = OUT / "eval"
for pairs in sorted(EVAL.glob("*_pairs.jsonl")):
    rows = [json.loads(line) for line in pairs.read_text(encoding="utf-8").split("\n") if line.strip()]
    assert all({"instruction", "response_a", "response_b", "mapping"} <= set(r) for r in rows), pairs
    print(f"{len(rows):>4}  {pairs.name}  ({rows[0]['candidate']} vs {rows[0]['baseline']})")

## 3. Pilot: a few pairs per file
Checks that the judge returns parseable verdicts before spending the full budget. Pilot judgments go to `eval_pilot/`,
which is never used for results.

In [ ]:
COMMON = ["--judge_model", JUDGE_MODEL, "--temperature", JUDGE_TEMPERATURE, "--backend", "kaggle", "--batch_size", BATCH_SIZE,
          "--sleep_seconds", SLEEP_SECONDS, "--max_retries", MAX_RETRIES] + ([] if INCLUDE_OPTIONAL else ["--skip_optional"])
if PILOT_PAIRS:
    PILOT = OUT.parent / "eval_pilot"
    PILOT.mkdir(parents=True, exist_ok=True)
    for pairs in EVAL.glob("*_pairs.jsonl"):
        shutil.copy2(pairs, PILOT / pairs.name)
    run(["python", "scripts/run_judging.py", "--eval_dir", PILOT, "--limit", PILOT_PAIRS, *COMMON])
    pilot = [json.loads(l) for f in sorted(PILOT.glob("*_judged.jsonl")) for l in f.read_text().split("\n") if l.strip()]
    ok = sum(r["status"] == "ok" for r in pilot)
    print(f"\nPilot: {ok}/{len(pilot)} verdicts parsed. Example:", json.dumps(pilot[0], indent=1)[:600] if pilot else None)
    assert ok >= 0.8 * len(pilot), "Most pilot verdicts failed - inspect eval_pilot/*_failures.jsonl before the full run."

## 4. Full judging (resumable; rerunning this cell only judges pairs without a valid verdict)

In [ ]:
start = time.time()
run(["python", "scripts/run_judging.py", "--eval_dir", EVAL, *COMMON])
print(f"Judging took {(time.time() - start) / 60:.1f} min")

## 5. Summaries, plots, results tables, download

In [ ]:
rows = []
for path in sorted(EVAL.glob("*_judged_summary.json")):
    s = json.loads(path.read_text())
    o = s["overall"]
    rows.append(f"| {path.stem.replace('_judged_summary', '')} | {s['rubric']} | {s['candidate']} vs {s['baseline']} | {o['n']} | "
                f"{o['wins']} | {o['ties']} | {o['losses']} | {o['unknown']} | {o['failures']} | "
                f"{o['win_rate_excluding_ties'] if o['win_rate_excluding_ties'] is None else round(o['win_rate_excluding_ties'], 3)} |")
display(Markdown("| comparison | rubric | candidate vs baseline | n | wins | ties | losses | unknown | failures | win rate (excl. ties) |\n"
                 "|---|---|---|---|---|---|---|---|---|---|\n" + "\n".join(rows)))
print("Judge model used:", JUDGE_MODEL, "| temperature:", JUDGE_TEMPERATURE)
run(["python", "scripts/plot_metrics.py", "--outputs", OUT])
run(["python", "scripts/summarize_results.py", "--outputs", OUT])
if (OUT / "plots" / "judge_results.png").exists():
    display(Image(filename=str(OUT / "plots" / "judge_results.png")))

bundle = Path("/kaggle/working/judge_bundle.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for sub in ("eval", "plots", "report"):
        for path in sorted((OUT / sub).rglob("*")):
            if path.is_file():
                zf.write(path, Path("outputs") / path.relative_to(OUT))
print(f"{bundle} {bundle.stat().st_size / 1e6:.1f} MB")
os.chdir("/kaggle/working")
display(FileLink(bundle.name))
os.chdir(repo)
print("Done. Save Version to preserve the judgments.")